# Project 1: Confidence Calibration of Small Open LLMs

**Research Question:** For small instruction-tuned LLMs (0.5B–7B), which confidence method best separates correct from wrong answers, and does the ranking change with model size?

- **Models:** `Qwen2.5-0.5B-Instruct`, `Qwen2.5-1.5B-Instruct`, `Qwen2.5-3B-Instruct` (fp16), `Qwen2.5-7B-Instruct` (4-bit NF4)
- **Datasets:** TriviaQA (500 open questions) + SciQ (500 multiple-choice questions)
- **Methods:** Token probability, Verbalized 0–100, P(True) self-check, Consistency (5 samples, T=0.7)
- **Estimated Runtime on a T4 GPU:** ~3 to 3.5 hours for all 4 models (estimate, labeled as estimate)
- **Outputs:** `out/*.jsonl`, `out/run_metadata.json`, and `p1_results.zip`
- **Resume support:** Partial results are saved after every question. If the session disconnects, rerun to resume without losing progress.

In [ ]:
# Cell 1: Install dependencies (Colab / Kaggle)
import os, sys
IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
IN_KAGGLE = "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if IN_COLAB or IN_KAGGLE:
    !pip install -q "transformers>=4.40.0" "datasets>=2.19.0" "accelerate>=0.30.0" "bitsandbytes>=0.43.0"
else:
    print("Local environment detected; skipping pip install.")

In [ ]:
# Cell 2: Configuration, optional Google Drive persistence, and SMOKE switch
import os
import sys
import gc
import json
import time
import math
import re
import random
import string
import zipfile
from pathlib import Path
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from datasets import load_dataset

SMOKE = os.environ.get("SMOKE") == "1"
SEED = 0
N_SAMPLES = 5 if SMOKE else 500
CONSISTENCY_SAMPLES = 2 if SMOKE else 5

# Optional Google Drive mount on Colab to protect progress if VM disconnects
USE_DRIVE = False  # Set to True on Colab if you want outputs saved straight to Drive
if IN_COLAB and USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = Path("/content/drive/MyDrive/p1_calibration_out")
        print(f"Google Drive mounted. Saving outputs to {OUT_DIR}")
    except Exception as e:
        print("Google Drive mount skipped/failed, using local out/ directory:", e)
        OUT_DIR = Path("out")
else:
    OUT_DIR = Path("out")

OUT_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("p1_results.zip")

if SMOKE:
    MODEL_CONFIGS = [
        {"model_id": "Qwen/Qwen2.5-0.5B-Instruct", "slug": "qwen2.5_0.5b_instruct", "load_4bit": False}
    ]
else:
    MODEL_CONFIGS = [
        {"model_id": "Qwen/Qwen2.5-0.5B-Instruct", "slug": "qwen2.5_0.5b_instruct", "load_4bit": False},
        {"model_id": "Qwen/Qwen2.5-1.5B-Instruct", "slug": "qwen2.5_1.5b_instruct", "load_4bit": False},
        {"model_id": "Qwen/Qwen2.5-3B-Instruct", "slug": "qwen2.5_3b_instruct", "load_4bit": False},
        {"model_id": "Qwen/Qwen2.5-7B-Instruct", "slug": "qwen2.5_7b_instruct", "load_4bit": True},
    ]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
print(f"SMOKE mode: {SMOKE} | Device: {device} ({gpu_name})")
print(f"Samples per dataset: {N_SAMPLES} | Consistency samples: {CONSISTENCY_SAMPLES}")
print(f"Output directory: {OUT_DIR.resolve()}")
print(f"Models: {[m['slug'] for m in MODEL_CONFIGS]}")

In [ ]:
# Cell 3: Answer Matchers (TriviaQA and SciQ)
def normalize_text(text: str) -> str:
    if not text:
        return ""
    text = text.lower()
    text = "".join(ch if ch not in set(string.punctuation) else " " for ch in text)
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    return " ".join(text.split())

def strip_answer_prefixes(text: str) -> str:
    clean = text.strip()
    prefixes = [
        r"^answer\s*:\s*",
        r"^the\s+answer\s+is\s*:?\s*",
        r"^it\s+is\s*:?\s*",
        r"^they\s+are\s*:?\s*",
        r"^it\'?s\s*:?\s*",
    ]
    for pattern in prefixes:
        clean = re.sub(pattern, "", clean, flags=re.IGNORECASE).strip()
    return clean

def match_trivia_qa(prediction: str, ground_truth_aliases: list) -> bool:
    if not prediction or not ground_truth_aliases:
        return False
    cleaned_pred = strip_answer_prefixes(prediction)
    norm_pred = normalize_text(cleaned_pred)
    if not norm_pred:
        return False
    norm_aliases = [normalize_text(a) for a in ground_truth_aliases if a and normalize_text(a)]
    for alias in norm_aliases:
        if norm_pred == alias:
            return True
    for alias in norm_aliases:
        if len(alias) >= 3 or len(alias.split()) > 1:
            if re.search(r"\b" + re.escape(alias) + r"\b", norm_pred):
                return True
    return False

def extract_sciq_letter(prediction: str):
    if not prediction:
        return None
    clean = prediction.strip()
    m = re.match(r"^\(?([A-D])\)?(?:[\.\:\)\s]|$)", clean, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m = re.search(r"\b(?:option|answer\s*is|choice)\s*:?\s*\(?([A-D])\)?\b", clean, flags=re.IGNORECASE)
    if m:
        return m.group(1).upper()
    m = re.search(r"\b([A-D])\b", clean)
    if m:
        return m.group(1).upper()
    return None

def match_sciq(prediction: str, correct_letter: str, correct_text: str = None) -> bool:
    if not prediction or not correct_letter:
        return False
    extracted = extract_sciq_letter(prediction)
    if extracted and extracted == correct_letter.strip().upper():
        return True
    if correct_text:
        norm_pred = normalize_text(strip_answer_prefixes(prediction))
        norm_correct = normalize_text(correct_text)
        if norm_pred and norm_pred == norm_correct:
            return True
    return False
print("Matchers loaded and verified.")

In [ ]:
# Cell 4: Load and Sample Data
print("Loading TriviaQA validation split...")
ds_trivia = load_dataset("mandarjoshi/trivia_qa", "rc.nocontext", split="validation")
rng_trivia = random.Random(SEED)
trivia_sample_indices = sorted(rng_trivia.sample(range(len(ds_trivia)), 500))
if SMOKE:
    trivia_sample_indices = trivia_sample_indices[:3]

sampled_trivia_data = []
for idx in trivia_sample_indices:
    row = ds_trivia[idx]
    ans = row["answer"]
    aliases = list(set([ans.get("value", "")] + ans.get("aliases", []) + ans.get("normalized_aliases", [])))
    aliases = [a.strip() for a in aliases if a and a.strip()]
    sampled_trivia_data.append({
        "dataset": "trivia_qa",
        "id": row["question_id"],
        "question": row["question"].strip(),
        "target": ans.get("value", "").strip(),
        "aliases": aliases,
    })

print("Loading SciQ test split...")
ds_sciq = load_dataset("allenai/sciq", split="test")
rng_sciq = random.Random(SEED)
sciq_sample_indices = sorted(rng_sciq.sample(range(len(ds_sciq)), 500))
if SMOKE:
    sciq_sample_indices = sciq_sample_indices[:2]

sampled_sciq_data = []
letters = ["A", "B", "C", "D"]
for idx in sciq_sample_indices:
    row = ds_sciq[idx]
    correct = row["correct_answer"].strip()
    distractors = [row["distractor1"].strip(), row["distractor2"].strip(), row["distractor3"].strip()]
    options = sorted([correct] + distractors)
    letter_to_opt = {letters[i]: options[i] for i in range(4)}
    gold_letter = [l for l, opt in letter_to_opt.items() if opt == correct][0]
    sampled_sciq_data.append({
        "dataset": "sciq",
        "id": f"sciq_test_{idx}",
        "question": row["question"].strip(),
        "choices": letter_to_opt,
        "correct_letter": gold_letter,
        "correct_text": correct,
    })

all_eval_items = sampled_trivia_data + sampled_sciq_data
print(f"Total items to evaluate per model: {len(all_eval_items)} (TriviaQA: {len(sampled_trivia_data)}, SciQ: {len(sampled_sciq_data)})")

In [ ]:
# Cell 5: Confidence extraction helper functions
def build_qa_messages(item: dict) -> list:
    if item["dataset"] == "trivia_qa":
        user_prompt = f"Answer the following question in as few words as possible. Output only the answer, nothing else.\nQuestion: {item['question']}\nAnswer:"
    else:
        choices = item["choices"]
        user_prompt = (
            f"Question: {item['question']}\n"
            f"(A) {choices['A']}\n"
            f"(B) {choices['B']}\n"
            f"(C) {choices['C']}\n"
            f"(D) {choices['D']}\n"
            f"Answer with only the letter corresponding to the correct answer: (A), (B), (C), or (D).\nAnswer:"
        )
    return [
        {"role": "system", "content": "You are a direct, concise question-answering assistant."},
        {"role": "user", "content": user_prompt},
    ]

def get_true_false_token_ids(tokenizer):
    true_ids, false_ids = set(), set()
    for w in ["True", " True", "true", " true"]:
        toks = tokenizer.encode(w, add_special_tokens=False)
        if len(toks) == 1:
            true_ids.add(toks[0])
    for w in ["False", " False", "false", " false"]:
        toks = tokenizer.encode(w, add_special_tokens=False)
        if len(toks) == 1:
            false_ids.add(toks[0])
    return list(true_ids), list(false_ids)

def evaluate_item(model, tokenizer, item, device, true_ids, false_ids, n_consistency_samples):
    messages = build_qa_messages(item)
    prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt_text, return_tensors="pt").to(device)
    input_len = inputs["input_ids"].shape[1]

    # 1. Greedy answer + Token probability
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            return_dict_in_generate=True,
            output_scores=True,
        )
    gen_token_ids = outputs.sequences[0][input_len:]
    answer_text = tokenizer.decode(gen_token_ids, skip_special_tokens=True).strip()

    log_probs = []
    for step_idx, step_logits in enumerate(outputs.scores):
        if step_idx >= len(gen_token_ids):
            break
        p = torch.softmax(step_logits[0], dim=-1)[gen_token_ids[step_idx]].item()
        log_probs.append(math.log(max(p, 1e-12)))
    token_prob = math.exp(sum(log_probs) / len(log_probs)) if log_probs else 0.5

    # Correctness
    if item["dataset"] == "trivia_qa":
        is_correct = match_trivia_qa(answer_text, item["aliases"])
        target_str = item["target"]
    else:
        is_correct = match_sciq(answer_text, item["correct_letter"], item["correct_text"])
        target_str = f"({item['correct_letter']}) {item['correct_text']}"

    # 2. Verbalized confidence (0-100)
    verb_msgs = [
        {"role": "system", "content": "You are an evaluator. Respond with only a single number between 0 and 100."},
        {"role": "user", "content": f"Question: {item['question']}\nAnswer: {answer_text}\nHow confident are you that this answer is correct? Respond with only a number between 0 and 100, where 0 means completely uncertain and 100 means completely certain."},
    ]
    verb_prompt = tokenizer.apply_chat_template(verb_msgs, tokenize=False, add_generation_prompt=True)
    verb_in = tokenizer(verb_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        verb_out = model.generate(**verb_in, max_new_tokens=10, do_sample=False)
    raw_verb = tokenizer.decode(verb_out[0][verb_in["input_ids"].shape[1]:], skip_special_tokens=True).strip()
    m = re.search(r"\b(100(?:\.0+)?|[1-9]?[0-9](?:\.[0-9]+)?)\b", raw_verb)
    conf_verbalized = float(m.group(1)) / 100.0 if m else 0.5
    verb_ok = m is not None

    # 3. P(True) self-check
    ptrue_msgs = [
        {"role": "system", "content": "You are an evaluator. Determine if the proposed answer is correct. Answer True or False."},
        {"role": "user", "content": f"Question: {item['question']}\nProposed Answer: {answer_text}\nIs this answer correct? Answer True or False.\nAnswer:"},
    ]
    ptrue_prompt = tokenizer.apply_chat_template(ptrue_msgs, tokenize=False, add_generation_prompt=True)
    ptrue_in = tokenizer(ptrue_prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        ptrue_logits = model(**ptrue_in).logits[0, -1, :]
        ptrue_probs = torch.softmax(ptrue_logits, dim=-1)
    p_t = sum(ptrue_probs[tid].item() for tid in true_ids)
    p_f = sum(ptrue_probs[fid].item() for fid in false_ids)
    conf_ptrue = (p_t / (p_t + p_f)) if (p_t + p_f) > 1e-12 else 0.5

    # 4. Consistency
    with torch.no_grad():
        sample_outs = model.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=True,
            temperature=0.7,
            top_p=0.95,
            num_return_sequences=n_consistency_samples,
        )
    samples = []
    matches = 0
    if item["dataset"] == "sciq":
        greedy_l = extract_sciq_letter(answer_text)
        for s_idx in range(n_consistency_samples):
            s_text = tokenizer.decode(sample_outs[s_idx][input_len:], skip_special_tokens=True).strip()
            samples.append(s_text)
            s_l = extract_sciq_letter(s_text)
            if greedy_l and s_l == greedy_l:
                matches += 1
            elif not greedy_l and normalize_text(s_text) == normalize_text(answer_text):
                matches += 1
    else:
        norm_g = normalize_text(strip_answer_prefixes(answer_text))
        for s_idx in range(n_consistency_samples):
            s_text = tokenizer.decode(sample_outs[s_idx][input_len:], skip_special_tokens=True).strip()
            samples.append(s_text)
            norm_s = normalize_text(strip_answer_prefixes(s_text))
            if norm_g and (norm_s == norm_g or match_trivia_qa(s_text, [answer_text])):
                matches += 1
    conf_consistency = float(matches / n_consistency_samples)

    def clean_f(v):
        return 0.5 if (v is None or math.isnan(v) or math.isinf(v)) else float(min(max(v, 0.0), 1.0))

    return {
        "dataset": item["dataset"],
        "id": item["id"],
        "question": item["question"],
        "target": target_str,
        "greedy_answer": answer_text,
        "is_correct": bool(is_correct),
        "conf_token_prob": clean_f(token_prob),
        "conf_verbalized": clean_f(conf_verbalized),
        "conf_p_true": clean_f(conf_ptrue),
        "conf_consistency": clean_f(conf_consistency),
        "raw_verbalized_text": raw_verb,
        "verbalized_parsed": bool(verb_ok),
        "consistency_samples": samples,
    }
print("Evaluation pipeline helper functions defined.")

In [ ]:
# Cell 6: Main Execution Loop across Models with Auto-Resume
start_time_iso = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

for config in MODEL_CONFIGS:
    m_id = config["model_id"]
    slug = config["slug"]
    out_path = OUT_DIR / f"{slug}.jsonl"

    print(f"\n{'='*60}")
    print(f"Processing model: {m_id} -> {out_path}")

    existing_keys = set()
    if out_path.exists():
        with open(out_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if line:
                    try:
                        entry = json.loads(line)
                        existing_keys.add((entry["dataset"], entry["id"]))
                    except Exception:
                        pass
        print(f"Found {len(existing_keys)} existing completed items in {out_path}. Resuming...")

    items_to_run = [it for it in all_eval_items if (it["dataset"], it["id"]) not in existing_keys]
    if not items_to_run:
        print(f"All items already completed for {slug}.")
        continue

    print(f"Loading tokenizer for {m_id}...")
    tok = AutoTokenizer.from_pretrained(m_id)
    true_ids, false_ids = get_true_false_token_ids(tok)

    print(f"Loading weights for {m_id} (4-bit={config['load_4bit']})...")
    if config["load_4bit"]:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
        )
        model = AutoModelForCausalLM.from_pretrained(
            m_id,
            quantization_config=bnb_config,
            device_map="auto",
        )
    else:
        dtype = torch.float16 if torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(m_id, dtype=dtype)
        model = model.to(device)
    model.eval()

    t_start_model = time.time()
    with open(out_path, "a", encoding="utf-8") as out_f:
        for idx, item in enumerate(items_to_run, 1):
            rec = evaluate_item(model, tok, item, device, true_ids, false_ids, CONSISTENCY_SAMPLES)
            rec["model_id"] = m_id
            out_f.write(json.dumps(rec) + "\n")
            out_f.flush()
            if idx % 25 == 0 or idx == len(items_to_run):
                elapsed = time.time() - t_start_model
                rate = idx / elapsed
                print(f"[{idx}/{len(items_to_run)}] items finished | {rate:.2f} items/s | Answer: {rec['greedy_answer'][:25]} | Correct: {rec['is_correct']}")

    del model
    del tok
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    print(f"Finished {slug} in {time.time() - t_start_model:.1f}s.")

In [ ]:
# Cell 7: Save Metadata and Zip Outputs
end_time_iso = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())

file_counts = {}
for f in sorted(OUT_DIR.glob("*.jsonl")):
    with open(f, "r", encoding="utf-8") as fp:
        file_counts[f.name] = sum(1 for _ in fp)

metadata = {
    "project": "p1_calibration_small_llms",
    "smoke_mode": SMOKE,
    "gpu": gpu_name,
    "torch_version": torch.__version__,
    "start_time": start_time_iso,
    "end_time": end_time_iso,
    "seed": SEED,
    "models": [m["model_id"] for m in MODEL_CONFIGS],
    "file_counts": file_counts,
}

meta_path = OUT_DIR / "run_metadata.json"
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zip_f:
    for path in sorted(OUT_DIR.glob("*")):
        zip_f.write(path, arcname=f"out/{path.name}")

print(f"\nSaved run metadata to {meta_path}:")
print(json.dumps(metadata, indent=2))
print(f"\nAll outputs zipped to: {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1024:.1f} KB)")

In [ ]:
# Cell 8: Colab Download Helper (optional)
try:
    from google.colab import files
    print("Triggering browser download for p1_results.zip...")
    files.download(str(ZIP_PATH))
except Exception:
    print(f"Running outside Colab. Results are ready at {ZIP_PATH.resolve()}")